# Import

In [8]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
import pandas as pd
import numpy as np

In [10]:
filepath = "/content/drive/MyDrive/data/enrollment/data/Enrollment.xlsx"

training_set = pd.read_excel(filepath,sheet_name='data')
holdout = pd.read_excel(filepath,sheet_name='holdhout')

# Convert to numeric

In [11]:
df = training_set.copy()

# FirstGen
df['FirstGen'] = df['FirstGen'].fillna(0).apply(lambda x: 0 if x == 0 else 1)

# Gender
df['Female'] = df['Gender'].apply(lambda x: 1 if x == 'F' else 0)
df['No Gender'] = df['Gender'].apply(lambda x: 1 if x == 'N' else 0)
df['Male'] = df['Gender'].apply(lambda x: 1 if x == 'M' else 0)
df = df.drop('Gender',axis=1)

# Citizen
df['Citizen'] = df['Citizen'].dropna().apply(lambda x: 1 if x == 'Y' else 0)

# StudentType
df['New Student'] = df['StudentType'].apply(lambda x: 1 if x == 5 else 0)
df = df.drop('StudentType',axis=1)

# Major
df = df.drop('MAJOR1',axis=1)
df['Double Major'] = df['MAJOR2'].fillna(0).apply(lambda x: 0 if x == 0 else 1)
df = df.drop('MAJOR2',axis=1)

# highschool_desc
df = pd.get_dummies(df, columns=['highschool_desc'], prefix='highschool')

# address
df = df.drop(['street','city'],axis=1)

# state
df['CA Resident'] = df['state'].dropna().apply(lambda x: 1 if x == 'CA' else 0)
df = df.drop(['state','zip'],axis=1)

# RELIGION
no_affiliation = ['No affliation/none listed']
sda = ['Seventh-day Adventist']
christian = ['Christian', 'Baptist', 'Lutheran', 'Latter-day Saints',
                  'Christian-Disc of Christ', 'Methodist', 'Presbyterian',
                  'First Christian', 'Protestant', 'Church of God',
                  'Evangelical', 'Pentecostal', 'Apostolic', 'Church of Christ',
                  'Nazarene', 'Assembly of God', 'African-Methodist-Episcopal',
                  'United Church of Christ', 'Anglican', 'Congregational',
                  'Southern Baptist']
catholic = ['Roman Catholic', 'Eastern Orthodox', 'Coptic Orthodox']
other = [
    'Agnostic', 'Jainism', 'Muslim', 'Sikh', 'Hindu', 'Buddhist',
    'Jehovahs Witnesses', 'Nichiren Shosshu of America', 'Episcopalian',
    'Unification', 'Friends/Quaker']

df['RELIGION_No_Affiliation'] = df['RELIGION'].apply(lambda x: 1 if x in no_affiliation else 0)
df['RELIGION_SDA'] = df['RELIGION'].apply(lambda x: 1 if x in sda else 0)
df['RELIGION_Christian'] = df['RELIGION'].apply(lambda x: 1 if x in christian else 0)
df['RELIGION_Catholic'] = df['RELIGION'].apply(lambda x: 1 if x in catholic else 0)
df['RELIGION_Other'] = df['RELIGION'].apply(lambda x: 1 if x in other else 0)
df = df.drop('RELIGION',axis=1)

# NETHN
df = pd.get_dummies(df, columns=['NETHN'], prefix='NETHN')

# RACE
unique_races = {'B', 'N', 'W', 'A', 'AI'} # Black, Native Hawaiian, White, Asian, American Indian
for race in unique_races:
    df[f'RACE_{race}'] = df['RACE'].dropna().apply(lambda x: 1 if race in x else 0)
df['RACE_unknown'] = df['RACE'].isna().astype(int)
df = df.drop('RACE', axis=1)

# GPA
df = df[df['HighSchool_GPA'] != 328]

# PERCENTILE
df = df.drop('HighSchool_PERCENTILE',axis=1) # drop as recommended by Dr. Karina

# CLASS_RANK
# --- nothing to change ---

# NumDays_app_adm
df = df[df['NumDays_app_adm'] >= 0]

# admit_deg_code
df['Pre Professional'] = df['admit_deg_code'].apply(lambda x: 1 if x == 3 else 0)
df = pd.get_dummies(df, columns=['admit_deg_code'], prefix = 'admit_deg_code')
df = df.drop(['admit_deg_code_49.0','admit_deg_code_50.0'],axis=1)

# drop not useful columns
df = df.drop(['Application Ineligible', 'Application Entered', 'Admitted', 'Enrolled', 'enrolled_fall_2024', 'enrolled_fall_2023', 'enrolled_fall_2022', 'enrolled_fall_2021'], axis=1)


# Fill NaN

In [12]:
print('pre fill NaN count')
print(df.isnull().sum().sum())

# dealing with Nans:
df['sat_S11'] = df['sat_S11'].fillna(0)
df['SAT_S12'] = df['SAT_S12'].fillna(0)
df = df[df['Citizen'].notna()]
df['HighSchool_GPA'] = df['HighSchool_GPA'].fillna(df['HighSchool_GPA'].median())
df['HighSchool_CLASS_RANK'] = df['HighSchool_CLASS_RANK'].fillna(df['HighSchool_CLASS_RANK'].median())
df = df[df['CA Resident'].notna()]
df['RACE_A'] = df['RACE_A'].fillna(0)
df['RACE_AI'] = df['RACE_AI'].fillna(0)
df['RACE_B'] = df['RACE_B'].fillna(0)
df['RACE_W'] = df['RACE_W'].fillna(0)
df['RACE_N'] = df['RACE_N'].fillna(0)
# fill sat_S11 with 0
# fill SAT_S12 with 0
# drop Citizen
# fill HighSchool_GPA with median
# fill HighSchool_CLASS_RANK with median
# drop CA Resident with unknown
# add an unknown race encoding

post_nan = df.isnull().sum().sum()
print('post fill NaN count')
print(post_nan)
if post_nan != 0:
  print(df.isnull().sum())

pre fill NaN count
13870
post fill NaN count
0


In [17]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4954 entries, 0 to 5381
Data columns (total 54 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   FirstGen                                   4954 non-null   int64  
 1   sat_S11                                    4954 non-null   float64
 2   SAT_S12                                    4954 non-null   float64
 3   Citizen                                    4954 non-null   float64
 4   HighSchool_GPA                             4954 non-null   float64
 5   HighSchool_CLASS_RANK                      4954 non-null   float64
 6   NumDays_app_adm                            4954 non-null   int64  
 7   Enrolled2                                  4954 non-null   int64  
 8   Female                                     4954 non-null   int64  
 9   No Gender                                  4954 non-null   int64  
 10  Male                         

In [18]:
df.to_csv('training_clean.csv')